In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)

In [14]:
data_path = "data/cleaned_sales_data.csv"
if not os.path.exists(data_path) and os.path.exists("../data/cleaned_sales_data.csv"):
    data_path = "../data/cleaned_sales_data.csv"

df = pd.read_csv(data_path, low_memory=False)

# 1. Remove outliers
df = df[(df["ClosePrice"] >= 100000) & (df["ClosePrice"] <= 5000000)].copy()

# 2. Convert dates
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

# 3. Create time-based splits
test_mask = (df["CloseDate"].dt.year == 2026) & (df["CloseDate"].dt.month == 5)
test_df = df[test_mask].copy()

train_window_months = 6
test_start = pd.Timestamp("2026-05-01")
train_start = test_start - pd.DateOffset(months=train_window_months)
train_df = df[(df["CloseDate"] >= train_start) & (df["CloseDate"] < test_start)].copy()

feature_cols = ["LivingArea_scaled", "BedroomsTotal_scaled", "BathroomsTotalInteger_scaled", "LotSizeSquareFeet_scaled"]
target_col = "ClosePrice"

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [15]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_r2 = r2_score(y_test, lr_preds)
print(f"Linear Regression Baseline R2: {lr_r2:.4f}")

Linear Regression Baseline R2: 0.3225


In [16]:
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

# Predict and score
dt_preds = dt_model.predict(X_test)
dt_r2 = r2_score(y_test, dt_preds)
print(f"Decision Tree Test R2: {dt_r2:.4f}")

Decision Tree Test R2: 0.3239


In [17]:
# Random Forest (using 100 trees)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_preds)
print(f"Random Forest Test R2: {rf_r2:.4f}")

Random Forest Test R2: 0.3703


In [18]:
results_df = pd.DataFrame({
    "Model": ["Linear Regression (Baseline)", "Decision Tree", "Random Forest"],
    "Test R2 Score": [lr_r2, dt_r2, rf_r2]
})

print("--- Model Comparison Framework Summary ---")
print(results_df.to_string(index=False))

--- Model Comparison Framework Summary ---
                       Model  Test R2 Score
Linear Regression (Baseline)       0.322464
               Decision Tree       0.323909
               Random Forest       0.370265
